## 🧩 Exercise 8: Comparative Analysis of Arabic and English Corpora

### **Objective**  
In this exercise, you will explore and compare the structural, lexical, and morphological characteristics of Arabic and English texts using **NLTK** and **Camel Tools**.

---

### **Step 1: Load Arabic and English Corpora**

1. Load the Arabic and the English sentences from the toy corpora provided in the notebook.   
2. What initial differences do you notice between Arabic and English sentences in terms of length, punctuation, and readability?



In [17]:
!pip install camel-tools nltk datasets pandas tqdm pyarabic -q
!camel_data -i light #all defaults light

No new packages will be installed.


In [18]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [19]:
# --- STEP 0: Install Dependencies ---
import os
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import FreqDist, pos_tag
from tqdm import tqdm
import numpy as np
import collections
import string
from camel_tools.utils.charsets import UNICODE_PUNCT_CHARSET

# --- Import Camel Tools Components ---
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.utils.dediac import dediac_ar
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tagger.default import DefaultTagger


# Initialize the Camel Tools POS Tagger once for efficiency
print("🐫 Initializing Camel Tools POS Tagger...")
mled = MLEDisambiguator.pretrained('calima-msa-r13')
ar_pos_tagger = DefaultTagger(mled, 'pos')
print("✅ Tagger ready.")



arabic_text = """
عِلْمُ الْحَاسُوبِ هُوَ دِرَاسَةُ الْحَوْسَبَةِ، وَالْأَتْمَتَةِ، وَالْمَعْلُومَاتِ.
خَوَارِزْمِيَّاتُ التَّعَلُّمِ الْآلِيِّ أَسَاسِيَّةٌ لِتَحْلِيلِ الْبَيَانَاتِ وَتَعَرُّفِ الْأَنْمَاطِ.
الْهَدَفُ مِنْ الذَّكَاءِ الِاصْطِنَاعِيِّ هُوَ إِنْشَاءُ أَنْظِمَةٍ يُمْكِنُهَا الِاسْتِدْلَالُ وَالتَّعَلُّمُ.
يَسْتَخْدِمُ التَّعَلُّمُ الْعَمِيقُ الشَّبَكَاتِ الْعَصَبِيَّةَ مُتَعَدِّدَةَ الطَّبَقَاتِ لِمُعَالَجَةِ الْبَيَانَاتِ الْمُعَقَّدَةِ.
هَيَاكِلُ الْبَيَانَاتِ مِثْلُ الْمَصْفُوفَاتِ وَالْقَوَائِمِ الْمُرْتَبِطَةِ هِيَ مَفَاهِيمُ أَسَاسِيَّةٌ فِي الْبَرْمَجَةِ.
تُوَفِّرُ الْحَوْسَبَةُ السَّحَابِيَّةُ وُصُولًا حَسَبَ الطَّلَبِ إِلَى مَوَارِدِ الْحَوْسَبَةِ عَبْرَ الْإِنْتَرْنِتِ.
الْأَمْنُ السِّيْبِرَانِيُّ حَاسِمٌ لِحِمَايَةِ الْأَنْظِمَةِ وَالْبَيَانَاتِ مِنْ الْهُجُومَاتِ الرَّقْمِيَّةِ.
يَرْبِطُ إِنْتَرْنِتُ الْأَشْيَاءِ الْأَغْرَاضَ الْيَوْمِيَّةَ بِشَبَكَةِ الْإِنْتَرْنِتِ.
يُشِيرُ الْبَيَانَاتُ الضَّخْمَةُ إِلَى مَجْمُوعَاتِ بَيَانَاتٍ كَبِيرَةٍ جِدًّا يُمْكِنُ تَحْلِيلُهَا حِسَابِيًّا.
يُتِيحُ مُعَالَجَةُ اللُّغَةِ الطَّبِيعِيَّةِ لِلْحَوَاسِيبِ فَهْمَ اللُّغَةِ الْبَشَرِيَّةِ.
يَتَطَلَّبُ تَدْرِيبُ نَمُوذَجِ التَّعَلُّمِ الْآلِيِّ مَجْمُوعَةَ بَيَانَاتٍ مُوَسَّمَةً كَبِيرَةً وَعَالِيَةَ الْجَوْدَةِ.
يَتَضَمَّnنُ التَّعَلُّمُ الْمُعَزَّزُ وَكِيلًا يَتَعَلَّمُ مِنْ خِلَالِ التَّجْرِبَةِ وَالْخَطَأِ.
تُقَاسُ كَفَاءَةُ الْخَوَارِزْمِيَّةِ بِتَعْقِيدِهَا الزَّمَنِيِّ وَالْمَكَانِيِّ.
يَسْتَخْدِمُ التَّعَلُّمُ الْمُرَاقَبُ بَيَانَاتِ مُدْخَلَاتٍ مُوَسَّمَةً لِتَدْرِيبِ النَّمُوذَجِ.
يُحَدِّدُ التَّعَلُّمُ غَيْرُ الْمُرَاقَبِ الْأَنْمَاطَ فِي الْبَيَانَاتِ غَيْرِ الْمُوَسَّمَةِ.
تَسْمَحُ الرُّؤْيَةُ الْحَاسُوبِيَّةُ لِلْآلَاتِ بِتَفْسِيرِ وَفَهْمِ الْمَعْلُومَاتِ الْبَصَرِيَّةِ.
يُتَرْجِمُ الْمُصَنِّفُ رَمْزَ الْبَرْمَجَةِ عَالِيَ الْمُسْتَوَى إِلَى رَمْزِ الْآلَةِ.
تُوَفِّرُ سَلَاسِلُ الْكُتَلِ سِجِلًّا لَامَرْكِزِيًّا وَغَيْرَ قَابِلٍ لِلتَّغْيِيرِ لِلْمُعَامَلَاتِ.
يَدْمِجُ عِلْمُ الرُّوبُوتَاتِ عِلْمَ الْحَاسُوبِ، وَالْهَنْدَسَةَ، وَالذَّكَاءَ الِاصْطِنَاعِيَّ.
يُرَكِّزُ الذَّكَاءُ الِاصْطِنَاعِيُّ الْأَخْلَاقِيُّ عَلَى تَطْوِيرِ أَنْظِمَةِ ذَكَاءٍ اِصْطِنَاعِيٍّ عَادِلَةٍ وَشَفَّافَةٍ.
اِخْتِبَارُ تُورِينْج هُوَ طَرِيقَةُ اسْتِفْسَارٍ لِتَحْدِيدِ مَا إِذَا كَانَ الْحَاسُوبُ قَادِرًا عَلَى التَّفْكِيرِ.
تَعِدُ الْحَوْسَبَةُ الْكَمِّيَّةُ بِحَلِّ بَعْضِ الْمَشَاكِلِ الْمُعَقَّدَةِ أَسْرَعَ بِكَثِيرٍ مِنَ الْحَوَاسِيبِ التَّقْلِيدِيَّةِ.
"""

english_text = """
Computer science is the study of computation, automation, and information.
Machine learning (ML) algorithms are essential for data analysis and pattern recognition.
The goal of Artificial Intelligence (AI) is to create systems that can reason and learn.
Deep learning utilizes neural networks with multiple layers to process complex data.
Data structures like arrays and linked lists are fundamental concepts in programming.
Cloud computing provides on-demand access to computing resources over the internet.
Cybersecurity is crucial for protecting systems and data from digital attacks.
The Internet of Things (IoT) connects everyday objects to the internet.
Big data refers to extremely large data sets that may be analyzed computationally.
Natural Language Processing (NLP) enables computers to understand human language.
Training an ML model requires a large, high-quality labeled dataset.
Reinforcement learning involves an agent learning through trial and error.
The efficiency of an algorithm is measured by its time and space complexity.
Supervised learning uses labeled input data to train the model.
Unsupervised learning identifies patterns in unlabeled data.
Computer vision allows machines to interpret and understand visual information.
A compiler translates high-level programming code into machine code.
Blockchains provide a decentralized and immutable ledger for transactions.
Robotics integrates computer science, engineering, and artificial intelligence.
Ethical AI focuses on developing AI systems that are fair and transparent.
The Turing test is a method of inquiry for determining if a computer is capable of thinking.
Quantum computing promises to solve certain complex problems much faster than classical computers.
"""



🐫 Initializing Camel Tools POS Tagger...
✅ Tagger ready.




### **Step 2: Sentence Segmentation and Tokenization**

1. Segment the **Arabic text** into sentences using:
   - `nltk.sent_tokenize()`
2. Print the number of sentences produced by the tokenizer.
3. Then, tokenize the sentences into **words** using both nltk and camel_tools tokenizers using their word-level tools.
   - Compute the **number of tokens** per sentence for both tokenizers.
4. **Question:**  
   What differences do you notice between the two tokenization methods?  
   Which one seems more accurate for Arabic?



In [20]:
arabic_sentences = sent_tokenize(arabic_text)
print(f"Number of sentences: {len(arabic_sentences)}")
arabic_words_nltk = []
arabic_words_camel = []
for sentence in arabic_sentences:
  arabic_words_nltk.extend(word_tokenize(sentence))
  arabic_words_camel.extend(simple_word_tokenize(sentence))
print(f"Number of tokens (NLTK): {len(arabic_words_nltk)}")
print(f"Number of tokens (Camel): {len(arabic_words_camel)}")

Number of sentences: 22
Number of tokens (NLTK): 227
Number of tokens (Camel): 231



### **Step 3: Lexical Statistics**

For both the Arabic and English corpora:

1. Count:
   - Total number of **words (tokens)**
   - Number of **unique words**
   - Average number of **words per sentence**
2. Compare the results between Arabic and English.
3. **Question:**  
   Which language shows a higher token variety (unique words)?  
   How might morphology explain this difference?



In [21]:
english_sentences = sent_tokenize(english_text)
english_words = []
for sentence in english_sentences:
  english_words.extend(word_tokenize(sentence))
print(f"Number of Arabic Words: {len(arabic_words_camel)}")
print(f"Number of English words: {len(english_words)}")
print(f"Number of unique Arabic words: {len(set(arabic_words_camel))}")
print(f"Number of unique English words: {len(set(english_words))}")
print(f"Average number of words per sentence (Arabic): {int(len(arabic_words_camel) / len(arabic_sentences))}")
print(f"Average number of words per sentence (English): {int(len(english_words) / len(english_sentences))}")

Number of Arabic Words: 231
Number of English words: 280
Number of unique Arabic words: 178
Number of unique English words: 171
Average number of words per sentence (Arabic): 10
Average number of words per sentence (English): 12




### **Step 4: Effect of Diacritics**

1. Remove **Arabic diacritics** from the Arabic corpus (using camel_tools).  
2. Recompute:
   - Total words
   - Unique words
   - Average words per sentence  
3. Compare the statistics before and after removing diacritics.
4. **Question:**  
   How does removing diacritics affect token counts and vocabulary diversity?



In [22]:
arabic_words_no_diacritics = []
for word in arabic_words_camel:
  arabic_words_no_diacritics.append(dediac_ar(word))
print(f"Number of Arabic Words (no diacritics): {len(arabic_words_no_diacritics)}")
print(f"Number of unique Arabic words (no diacritics): {len(set(arabic_words_no_diacritics))}")
print(f"Average number of words per sentence (Arabic, no diacritics): {int(len(arabic_words_no_diacritics) / len(arabic_sentences))}")

Number of Arabic Words (no diacritics): 231
Number of unique Arabic words (no diacritics): 162
Average number of words per sentence (Arabic, no diacritics): 10



### **Step 5: Sentence and Word Length Analysis**

For both Arabic and English corpora:

1. Compute:
   - Average **sentence length** (in characters and in tokens)
   - Average **word length** (in characters)
2. Compare the two languages statistics.
3. **Question:**  
   What do these differences tell you about the structure of Arabic vs. English writing?

4. **POS Tagging (Arabic Only):**  
   - Use `camel_tools` tagger to perform POS tagging on the Arabic sentences.  
   - Examine whether the predicted tags are accurate and meaningful.  


In [23]:
print(f"Average sentence length (Arabic): {int(np.mean([len(sentence) for sentence in arabic_sentences]))}")
print(f"Average sentence length (English): {int(np.mean([len(sentence) for sentence in english_sentences]))}")

print(f"Average word length (Arabic): {int(np.mean([len(word) for word in arabic_words_camel]))}")
print(f"Average word length (English): {int(np.mean([len(word) for word in english_words]))}")

arabic_tags= ar_pos_tagger.tag(arabic_words_camel)
print(arabic_tags)

# make tags meaningful by grouping them with the words associated
arabic_tagged_words = {}
for word,tag in zip(arabic_words_camel,arabic_tags):
  if word not in arabic_tagged_words:
    arabic_tagged_words[word] = tag
print("More Meaningful tags:")
print(arabic_tagged_words)


Average sentence length (Arabic): 107
Average sentence length (English): 78
Average word length (Arabic): 9
Average word length (English): 5
['verb', 'noun', 'pron', 'noun', 'noun_prop', 'punc', 'noun_prop', 'punc', 'noun', 'punc', 'noun_prop', 'noun', 'adj', 'adj', 'noun', 'noun', 'verb', 'noun', 'punc', 'noun', 'prep', 'noun', 'adj', 'pron', 'noun', 'noun', 'verb', 'noun', 'noun', 'punc', 'verb', 'noun', 'adj', 'noun', 'adj', 'adj', 'noun', 'noun', 'noun', 'adj', 'punc', 'noun', 'noun', 'noun', 'noun', 'noun', 'adj', 'pron', 'noun', 'adj', 'prep', 'noun', 'punc', 'verb', 'noun_prop', 'noun_prop', 'noun', 'noun', 'noun', 'prep', 'noun', 'noun_prop', 'noun', 'noun_prop', 'punc', 'noun', 'noun_prop', 'adj', 'noun', 'noun', 'noun', 'prep', 'noun', 'adj', 'punc', 'verb', 'noun_prop', 'noun', 'noun', 'adj', 'noun', 'noun_prop', 'punc', 'verb', 'noun', 'adj', 'prep', 'noun', 'noun', 'adj', 'noun', 'verb', 'noun', 'adj', 'punc', 'verb', 'noun', 'noun', 'adj', 'noun', 'pron', 'noun', 'adj', '



### **Step 6: Part-of-Speech Frequency Comparison**

For both corpora:

1. Identify and count the frequency of:
   - **Nouns**
   - **Verbs**
   - **Adjectives**
2. Print the **top 10 most frequent** ones in each category.
3. Compare Arabic vs. English results.

**Questions:**
- Which language shows a higher proportion of nouns vs. verbs (A higher ratio indicates a prevalence of nouns, while a lower ratio suggests a greater proportion of verbs)?
- Do adjectives appear more frequently in English than Arabic?


In [24]:
verbs_ar , nouns_ar, adjs_ar = [],[],[]
for word,tag in arabic_tagged_words.items():
  if tag.startswith("n"):
    nouns_ar.append(word)
  elif tag.startswith("v"):
    verbs_ar.append(word)
  elif tag.startswith("a"):
    adjs_ar.append(word)
print(f"Number of nouns (Arabic): {len(nouns_ar)}")
print(f"Number of verbs (Arabic): {len(verbs_ar)}")
print(f"Number of adjectives (Arabic): {len(adjs_ar)}")

english_tagged_words = pos_tag(english_words)
verbs_en , nouns_en, adjs_en = [],[],[]
for word,tag in english_tagged_words:
  if tag.startswith("N"):
   nouns_en.append(word)
  elif tag.startswith("V"):
    verbs_en.append(word)
  elif tag.startswith("J"):
    adjs_en.append(word)
print(f"Number of nouns (English): {len(nouns_en)}")
print(f"Number of verbs (English):{len(verbs_en)} ")
print(f"Number of adjectives (English): {len(adjs_en)}")

print(f"Top 10 nouns (Arabic):{collections.Counter(nouns_ar).most_common(10)}")
print(f"Top 10 verbs (Arabic):{collections.Counter(verbs_ar).most_common(10)}")
print(f"Top 10 adjectives (Arabic):{collections.Counter(adjs_ar).most_common(10)}")

print(f"Top 10 nouns (English):{collections.Counter(nouns_en).most_common(10)}")
print(f"Top 10 verbs (English):{collections.Counter(verbs_en).most_common(10)}")
print(f"Top 10 adjectives (English):{collections.Counter(adjs_en).most_common(10)}")

Number of nouns (Arabic): 109
Number of verbs (Arabic): 21
Number of adjectives (Arabic): 34
Number of nouns (English): 101
Number of verbs (English):47 
Number of adjectives (English): 28
Top 10 nouns (Arabic):[('الْحَاسُوبِ', 1), ('دِرَاسَةُ', 1), ('الْحَوْسَبَةِ', 1), ('وَالْأَتْمَتَةِ', 1), ('وَالْمَعْلُومَاتِ', 1), ('خَوَارِزْمِيَّاتُ', 1), ('التَّعَلُّمِ', 1), ('لِتَحْلِيلِ', 1), ('الْبَيَانَاتِ', 1), ('الْأَنْمَاطِ', 1)]
Top 10 verbs (Arabic):[('عِلْمُ', 1), ('وَتَعَرُّفِ', 1), ('يُمْكِنُهَا', 1), ('يَسْتَخْدِمُ', 1), ('تُوَفِّرُ', 1), ('يَرْبِطُ', 1), ('يُشِيرُ', 1), ('يُمْكِنُ', 1), ('يُتِيحُ', 1), ('يَتَطَلَّبُ', 1)]
Top 10 adjectives (Arabic):[('الْآلِيِّ', 1), ('أَسَاسِيَّةٌ', 1), ('الِاصْطِنَاعِيِّ', 1), ('الْعَمِيقُ', 1), ('الْعَصَبِيَّةَ', 1), ('مُتَعَدِّدَةَ', 1), ('الْمُعَقَّدَةِ', 1), ('الْمُرْتَبِطَةِ', 1), ('حَاسِمٌ', 1), ('الرَّقْمِيَّةِ', 1)]
Top 10 nouns (English):[('data', 7), ('AI', 3), ('systems', 3), ('Computer', 2), ('science', 2), ('information', 2), ('lear



### **Reflection**

- How do tokenization and diacritics influence Arabic text analysis?
- How might these findings guide NLP preprocessing decisions for multilingual systems?


# ✍️ Exercise 9: Extractive Summarization of Arabic Texts

## 🎯 Objective
In this exercise, you will build a **basic extractive summarizer** for **Arabic scientific texts**.  
Sentences will be scored according to the frequency of important words — specifically **nouns** and **verbs** — and the highest-scoring sentences will form the summary.  
The exercise relies on **NLTK** for sentence segmentation and **Camel Tools** for morphological analysis.

---

## 📚 Corpus: Computer Science Topics
A small Arabic corpus of **six paragraphs** about various computer-science topics is provided.  
Treat **each paragraph as a separate document** to be summarized.






In [25]:

import nltk
import math
from camel_tools.utils.dediac import dediac_ar
from camel_tools.tokenizers.word import simple_word_tokenize
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tagger.default import DefaultTagger


print("🐫 Initializing Camel Tools POS Tagger...")
mled = MLEDisambiguator.pretrained('calima-msa-r13')
ar_pos_tagger = DefaultTagger(mled, 'pos')
print("✅ Tagger ready.")

# --- Provided Corpus ---
corpus = """
الذَّكَاءُ الِاصْطِنَاعِيُّ (AI) هُوَ فَرْعٌ مِنْ عُلُومِ الْحَاسُوبِ يُحَاكِي الْقُدُرَاتِ الذِّهْنِيَّةَ الْبَشَرِيَّةَ. يَهْدِفُ إِلَى تَطْوِيرِ أَنْظِمَةٍ قَادِرَةٍ عَلَى التَّعَلُّمِ وَالِاسْتِنْتَاجِ وَاتِّخَاذِ الْقَرَارَاتِ. يَنْقَسِمُ الذَّكَاءُ الِاصْطِنَاعِيُّ إِلَى فِئَتَيْنِ رَئِيسِيَّتَيْنِ: الضَّيِّقُ وَالْعَامُّ. تَتَفَوَّقُ الْأَنْظِمَةُ الضَّيِّقَةُ فِي مَهَامٍّ مُحَدَّدَةٍ، مِثْلَ التَّعَرُّفِ عَلَى الصُّوَرِ. أَمَّا الذَّكَاءُ الْعَامُّ فَيَسْعَى لِمُحَاكَاةِ الْفَهْمِ الْبَشَرِيِّ الشَّامِلِ. تُعَدُّ خَوَارِزْمِيَّاتُ التَّعَلُّمِ الْآلِيِّ أَسَاسَ مُعْظَمِ تَطْبِيقَاتِ الذَّكَاءِ الِاصْطِنَاعِيِّ الْحَدِيثَةِ.

الْأَمْنُ السِّيبْرَانِيُّ هُوَ مَجْمُوعَةُ التَّقْنِيَاتِ وَالْعَمَلِيَّاتِ الْمُصَمَّمَةِ لِحِمَايَةِ الشَّبَكَاتِ وَالْأَجْهِزَةِ مِنَ الْهُجُومَاتِ الرَّقْمِيَّةِ. تَسْتَهْدِفُ هَذِهِ الْهُجُومَاتُ عَادَةً الْوُصُولَ إِلَى الْمَعْلُومَاتِ الْحَسَّاسَةِ أَوْ تَغْيِيرَهَا أَوْ إِتْلَافَهَا. يَعْتَمِدُ الْخُبَرَاءُ عَلَى اسْتِرَاتِيجِيَّاتٍ مُتَعَدِّدَةٍ لِتَأْمِينِ الْبِنْيَةِ التَّحْتِيَّةِ الرَّقْمِيَّةِ. تَتَضَمَّنُ هَذِهِ الِاسْتِرَاتِيجِيَّاتُ تَشْفِيرَ الْبَيَانَاتِ وَاسْتِخْدَامَ جُدُرَانِ الْحِمَايَةِ. يَتَطَلَّبُ الْأَمْنُ الْفَعَّالُ تَوْعِيَةَ الْمُسْتَخْدِمِينَ بِالْمَخَاطِرِ الْمُحْتَمَلَةِ. يُشَكِّلُ التَّصَيُّدُ الِاحْتِيَالِيُّ وَالْبَرْمَجِيَّاتُ الْخَبِيثَةُ أَكْبَرَ التَّهْدِيدَاتِ الْحَالِيَّةِ.

الْحَوْسَبَةُ السَّحَابِيَّةُ (Cloud Computing) هِيَ نَمُوذَجٌ لِتَقْدِيمِ خَدَمَاتِ الْحَوْسَبَةِ عَبْرَ شَبَكَةِ الْإِنْتَرْنِتِ. تُتِيحُ هَذِهِ الْخَدَمَاتُ لِلشَّرِكَاتِ الْوُصُولَ إِلَى مَوَارِدَ مِثْلَ التَّخْزِينِ وَقُوَّةِ الْمُعَالَجَةِ دُونَ الْحَاجَةِ لِإِدَارَةِ بِنْيَةٍ تَحْتِيَّةٍ خَاصَّةٍ. تَنْقَسِمُ الْخَدَمَاتُ السَّحَابِيَّةُ إِلَى ثَلَاثِ فَئَاتٍ: الْبِنْيَةُ التَّحْتِيَّةُ كَخِدْمَةٍ (IaaS)، وَالْمِنَصَّةُ كَخِدْمَةٍ (PaaS)، وَالْبَرْمَجِيَّاتُ كَخِدْمَةٍ (SaaS). يُوَفِّرُ هَذَا النَّمُوذَجُ مَرُونَةً كَبِيرَةً وَيُقَلِّلُ مِنَ التَّكَالِيفِ التَّشْغِيلِيَّةِ. تَعْتَمِدُ الشَّرِكَاتُ الْكُبْرَى مِثْلُ أَمَازُونَ وَمَايْكْرُوسُوفْتَ عَلَى تَقْدِيمِ هَذِهِ الْحُلُولِ.

عِلْمُ الْبَيَانَاتِ هُوَ مَجَالٌ مُتَعَدِّدُ التَّخَصُّصَاتِ يَدْمِجُ بَيْنَ الْإِحْصَاءِ وَعُلُومِ الْحَاسُوبِ لِاسْتِخْرَاجِ الرُّؤَى مِنْ مَجْمُوعَاتِ الْبَيَانَاتِ الْكَبِيرَةِ. يَقُومُ عُلَمَاءُ الْبَيَانَاتِ بِتَحْلِيلِ الْبَيَانَاتِ لِتَحْدِيدِ الْأَنْمَاطِ وَالتَّنَبُّؤِ بِالْأَحْدَاثِ الْمُسْتَقْبَلِيَّةِ. تَتَضَمَّنُ دَوْرَةُ حَيَاةِ مَشْرُوعِ عِلْمِ الْبَيَانَاتِ جَمْعَ الْبَيَانَاتِ وَتَنْظِيفَهَا وَتَحْلِيلَهَا. تُسْتَخْدَمُ لُغَاتُ بَرْمَجَةٍ مِثْلُ بَايْثُونَ وَآر (R) بِشَكْلٍ وَاسِعٍ فِي هَذَا الْمَجَالِ. تُسَاهِمُ نَمَاذِجُ التَّعَلُّمِ الْآلِيِّ فِي بِنَاءِ أَنْظِمَةٍ تَنْبُؤِيَّةٍ دَقِيقَةٍ. يُسَاعِدُ هَذَا الْعِلْمُ الشَّرِكَاتِ عَلَى اتِّخَاذِ قَرَارَاتٍ أَفْضَلَ.

تِقْنِيَةُ الْبُلُوكْتِشَيْن (Blockchain) هِيَ سِجِلُّ مُعَامَلَاتٍ رَقْمِيٌّ مُوَزَّعٌ وَلَامَرْكَزِيٌّ. تُعْرَفُ هَذِهِ التِّقْنِيَةُ بِدَوْرِهَا الْأَسَاسِيِّ فِي الْعُمْلَاتِ الْمُشَفَّرَةِ مِثْلِ الْبِتْكُويْن. تَتَكَوَّنُ سِلْسِلَةُ الْكُتَلِ مِنْ كُتَلٍ مُتَرَابِطَةٍ، كُلُّ كُتْلَةٍ تَحْتَوِي عَلَى طَابَعٍ زَمَنِيٍّ وَبَيَانَاتِ الْمُعَامَلَاتِ. يَجْعَلُ التَّشْفِيرُ وَالطَّبِيعَةُ الْمُوَزَّعَةُ مِنَ الصَّعْبِ جِدًّا تَغْيِيرَ الْبَيَانَاتِ الْمُسَجَّلَةِ. تُوَفِّرُ الْبُلُوكْتِشَيْنُ الشَّفَافِيَّةَ وَالْأَمَانَ فِي الْعَدِيدِ مِنَ التَّطْبِيقَاتِ. يُمْكِنُ اسْتِخْدَامُهَا فِي سَلَاسِلِ التَّوْرِيدِ وَالتَّصْوِيتِ الْإِلِكْتُرُونِيِّ وَالْعُقُودِ الذَّكِيَّةِ.

الْحَوْسَبَةُ الْكَمِّيَّةُ هِيَ نَوْعٌ جَدِيدٌ مِنَ الْحَوْسَبَةِ يَعْتَمِدُ عَلَى مَبَادِئِ مِيكَانِيكَا الْكَمِّ. تَسْتَخْدِمُ الْحَوَاسِيبُ الْكَمِّيَّةُ "الْكُيُوبِت" (qubit) لِتَمْثِيلِ الْمَعْلُومَاتِ، بِخِلَافِ "الْبِتِّ" الثُّنَائِيِّ فِي الْحَوْسَبَةِ التَّقْلِيدِيَّةِ. يُمْكِنُ لِلْكُيُوبِت أَنْ يَكُونَ فِي حَالَاتٍ مُتَعَدِّدَةٍ فِي وَقْتٍ وَاحِدٍ بِفَضْلِ ظَاهِرَةِ التَّرَاكُبِ. تُتِيحُ هَذِهِ الْقُدْرَةُ لِلْحَوَاسِيبِ الْكَمِّيَّةِ حَلَّ بَعْضِ الْمَسَائِلِ الْمُعَقَّدَةِ بِسُرْعَةٍ تَفُوقُ الْحَوَاسِيبَ الْكَلاسِيكِيَّةَ. تَشْمَلُ تَطْبِيقَاتُهَا الْمُحْتَمَلَةُ تَطْوِيرَ الْأَدْوِيَةِ وَاكْتِشَافَ الْمَوَادِّ الْجَدِيدَةِ. لَا تَزَالُ هَذِهِ التِّقْنِيَةُ فِي مَرَاحِلِهَا الْأُولَى مِنَ التَّطْوِيرِ.
"""


🐫 Initializing Camel Tools POS Tagger...
✅ Tagger ready.



## 🧩 Step 1: Load and Segment Documents
1. Load the Arabic corpus.  
2. Split it into **six documents** (one per paragraph).  
3. For each document, use sentence segmentation to divide the text into sentences.  
4. **Print the number of sentences** found in the **first document**.



In [36]:
splitted_corpus = corpus.split("\n\n")
print(f"number of documents after split: {len(splitted_corpus)}")
sentences = []
for paragraph in splitted_corpus:
  sentences.append(sent_tokenize(paragraph))
print(f"number of sentences in the first document: {len(sentences[0])}")

number of documents after split: 6
number of sentences in the first document: 6


---

## ⚙️ Step 2: Preprocessing and Feature Extraction
For every document:

1. **Remove diacritics** from the full text.  
2. **Tokenize** the dediacritized text into words.  
3. Apply **part-of-speech tagging** using Camel Tools.  
4. Identify all unique **nouns** (`noun`, `noun_prop`) and **verbs** (`verb`).  
5. Combine these into a single **set** representing the document’s important words.

**Question:**  
How many unique nouns and verbs are found in the **first document** (on Artificial Intelligence)?


In [41]:
# remove diacs, tokenize
paragraph_tokens = []
for paragraph in splitted_corpus:
    paragraph = dediac_ar(paragraph)
    paragraph_tokens.append(simple_word_tokenize(paragraph))

# POS Tagging
paragraph_tags = []
for paragraph in paragraph_tokens:
    paragraph_tags.append(ar_pos_tagger.tag(paragraph))

paragraph_tokens_tagged = []
for i in range(len(paragraph_tokens)):
    current_paragraph = []
    for word,tag in zip(paragraph_tokens[i],paragraph_tags[i]):
        current_paragraph.append((word,tag))
    paragraph_tokens_tagged.append(current_paragraph)

# find unique nouns and verbs
def get_unique_nouns_and_verbs(paragraph):
  nouns = set()
  verbs = set()
  for word,tag in paragraph:
    if tag.startswith("n"):
      nouns.add(word)
    elif tag.startswith("v"):
      verbs.add(word)
  return (nouns,verbs)
important_words = set()
for i in range(len(splitted_corpus)):
  nouns,verbs = get_unique_nouns_and_verbs(paragraph_tokens_tagged[i])
  important_words.update(nouns)
  important_words.update(verbs)

print(important_words)


{'لحماية', 'يدمج', 'تغيير', 'بخلاف', 'تفوق', 'لتقديم', 'التشفير', 'علماء', 'برمجة', 'المعاملات', 'التطبيقات', 'يجعل', 'الحاسوب', 'البنية', 'الأنماط', 'وتحليلها', 'أساس', 'والأجهزة', 'التخصصات', 'بدورها', 'الكبرى', 'حالات', 'والأمان', 'يمكن', 'يوفر', 'شبكة', 'بفضل', 'مجموعات', 'بنية', 'والاستنتاج', 'مجموعة', 'الأدوية', 'تتفوق', 'بين', 'والطبيعة', 'تشمل', 'مهام', 'التخزين', 'ومايكروسوفت', 'الذهنية', 'الخدمات', 'دقيقة', 'للشركات', 'وآر', 'التطوير', 'بناء', 'تعتمد', 'فئات', 'لاستخراج', 'تعرف', 'الأنظمة', 'بشكل', 'فئتين', 'فرع', 'سلسلة', 'واكتشاف', 'كتل', 'الحلول', 'نماذج', 'والتنبؤ', 'العلم', 'الخبراء', 'عادة', 'وقت', 'الكمية', 'بتحليل', 'جدران', 'مرونة', 'التعرف', 'خدمات', 'تتيح', 'الحماية', 'حياة', 'يحاكي', 'أمازون', 'الحاجة', 'يساعد', 'الكم', 'الأمن', 'الذكاء', 'يعتمد', 'السحابية', 'تطوير', 'يهدف', 'كخدمة', 'مشروع', 'جدا', 'لمحاكاة', 'تطبيقات', 'وبيانات', 'الهجومات', 'الوصول', 'الصور', 'المعالجة', 'والعمليات', 'والمنصة', 'البلوكتشين', 'مبادئ', 'يقوم', 'سجل', 'السيبراني', 'اتخاذ', 'دون',



## 🧮 Step 3: Sentence Scoring
For each sentence in a document:

1. Preprocess the sentence (remove diacritics and tokenize).  
2. Count how many of the document’s important nouns and verbs occur in the sentence.  
3. Assign a **score** equal to that count.  
   - A higher score → the sentence is more central to the document’s theme.

**Question:**  
For the **first document**, list each sentence alongside its computed score.



In [53]:
all_sentences_scored = []

for doc_idx, paragraph in enumerate(splitted_corpus):
  doc_sentences = sent_tokenize(paragraph)
  for sent_idx , sentence in enumerate(doc_sentences):
    sentence = dediac_ar(sentence)
    sentence_tokens = simple_word_tokenize(sentence)
    score = sum([1 for word in sentence_tokens if word in important_words])
    all_sentences_scored.append((doc_idx,sent_idx,sentence,score))
    #only print first doc
    if doc_idx == 0:
      print(f"Sentence {sent_idx} score : {score}")

Sentence 0 score : 7
Sentence 1 score : 7
Sentence 2 score : 4
Sentence 3 score : 6
Sentence 4 score : 4
Sentence 5 score : 7


---

## 📝 Step 4: Summary Generation
1. Choose a **summary percentage** (e.g., 40%).  
2. For each document:  
   - Compute the number of sentences to include:  
     ⌈ *number of sentences × summary percentage* ⌉  
     Example: 8 sentences × 0.40 = 3.2 → keep 4 sentences.  
   - Rank all sentences by score (highest → lowest).  
   - Select the top-ranked sentences.  
   - **Preserve their original order** when forming the summary.  
3. For the **first two documents**, display both the **original text** and the **generated summary** for comparison.

---

In [58]:
percent = 0.4
num_sentences = []
for doc_idx, paragraph in enumerate(splitted_corpus):
 num_sentences.append(int(percent * len(sent_tokenize(paragraph))))

all_sentences_sorted= sorted(all_sentences_scored, key=lambda x: x[3], reverse=True)

top_ranks = [[] for _ in range(len(splitted_corpus))]
summaries = ["" for _ in range(len(splitted_corpus))]
for doc_idx, sent_idx, sentence, score in all_sentences_sorted:
    if len(top_ranks[doc_idx]) < num_sentences[doc_idx]:
        top_ranks[doc_idx].append((doc_idx, sent_idx, sentence, score))

for doc_idx in range(len(splitted_corpus)):
    # Sort selected sentences by their original order
    top_sentences_sorted = sorted(top_ranks[doc_idx], key=lambda x: x[1])
    # Extract just the sentence text
    sentence_texts = [sent[2] for sent in top_sentences_sorted]
    # Join into summary
    summaries[doc_idx] = " ".join(sentence_texts)

print(f"Original Text: {splitted_corpus[0]}")
print(f"Summary: {summaries[0]}")


Original Text: 
الذَّكَاءُ الِاصْطِنَاعِيُّ (AI) هُوَ فَرْعٌ مِنْ عُلُومِ الْحَاسُوبِ يُحَاكِي الْقُدُرَاتِ الذِّهْنِيَّةَ الْبَشَرِيَّةَ. يَهْدِفُ إِلَى تَطْوِيرِ أَنْظِمَةٍ قَادِرَةٍ عَلَى التَّعَلُّمِ وَالِاسْتِنْتَاجِ وَاتِّخَاذِ الْقَرَارَاتِ. يَنْقَسِمُ الذَّكَاءُ الِاصْطِنَاعِيُّ إِلَى فِئَتَيْنِ رَئِيسِيَّتَيْنِ: الضَّيِّقُ وَالْعَامُّ. تَتَفَوَّقُ الْأَنْظِمَةُ الضَّيِّقَةُ فِي مَهَامٍّ مُحَدَّدَةٍ، مِثْلَ التَّعَرُّفِ عَلَى الصُّوَرِ. أَمَّا الذَّكَاءُ الْعَامُّ فَيَسْعَى لِمُحَاكَاةِ الْفَهْمِ الْبَشَرِيِّ الشَّامِلِ. تُعَدُّ خَوَارِزْمِيَّاتُ التَّعَلُّمِ الْآلِيِّ أَسَاسَ مُعْظَمِ تَطْبِيقَاتِ الذَّكَاءِ الِاصْطِنَاعِيِّ الْحَدِيثَةِ.
Summary: 
الذكاء الاصطناعي (AI) هو فرع من علوم الحاسوب يحاكي القدرات الذهنية البشرية. يهدف إلى تطوير أنظمة قادرة على التعلم والاستنتاج واتخاذ القرارات.




## 💭 Reflection
- What are the **main limitations** of this frequency-based approach to summarization?  
- How might you **improve** the sentence-scoring method to achieve higher-quality summaries?
